# 2. The sequence model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SMLCI/acia-core/blob/main/docs/tutorials/02_the_sequence_model.ipynb)

Everything in `acia` that produces or consumes images speaks one type:
{class}`~acia.base.ImageSequenceSource`. Learn its shape conventions and its
indexing rules once, and every reader, every renderer and every processor
behaves the same way.

This notebook uses a **synthetic sequence** so it runs instantly and you can see
exactly what goes in and what comes out.

In [ ]:
# On Colab (or any fresh environment) this installs acia.
# Locally, if you already have acia installed, it is a no-op.
try:
    import acia  # noqa: F401
except ImportError:
    %pip install -q acia

## THWC

`acia` lays out image sequences as **`(T, H, W, C)`** — time, height, width,
channel — with the **channel axis last**.

There is no `Z`. `acia` is a 2D+t library: it models a time-lapse of a single
focal plane, which is what live-cell imaging in a microfluidic chip or on a
coverslip actually produces.

{class}`~acia.segm.local.THWCSequenceSource` wraps a numpy array in that layout
and is the simplest source there is. It validates the shape, so a mistake is
caught at construction rather than three steps later.

In [ ]:
import numpy as np

from acia.segm.local import THWCSequenceSource

rng = np.random.default_rng(0)

T, H, W, C = 24, 128, 160, 1
stack = np.zeros((T, H, W, C), dtype=np.uint8)

# a bright disc drifting diagonally across the field of view
yy, xx = np.mgrid[0:H, 0:W]
for t in range(T):
    cy, cx = 30 + 2.5 * t, 30 + 4.0 * t
    disc = ((yy - cy) ** 2 + (xx - cx) ** 2) < 12**2
    stack[t, ..., 0] = (disc * 200).astype(np.uint8)

stack += rng.integers(0, 25, size=stack.shape, dtype=np.uint8)

src = THWCSequenceSource(stack)
src

Pass an array that is not 4-dimensional and you get told immediately:

In [ ]:
try:
    THWCSequenceSource(np.zeros((10, 64, 64)))  # missing the channel axis
except ValueError as err:
    print("ValueError:", err)

## Size and iteration

A source is `Sized` and `Iterable`: `len()` is the number of timepoints, and
iterating yields one frame at a time — lazily, so a long sequence never has to
fit in memory.

In [ ]:
print("len(src)  ", len(src))
print("size_t    ", src.size_t)
print("size_h    ", src.size_h)
print("size_w    ", src.size_w)
print("size_c    ", src.size_c)
print("channels  ", src.num_channels)

mean_per_frame = [float(np.asarray(f.raw).mean()) for f in src]
print("first 5 frame means:", [round(m, 1) for m in mean_per_frame[:5]])

## Indexing: integers give frames, slices give views

Indexing follows numpy over the same four axes. The one rule to remember:

* an **integer** on the time axis returns *that frame* (a `BaseImage`);
* a **slice or list** returns a *new source* — a lazy view.

In [ ]:
frame = src[5]  # integer -> one frame
view = src[5:15]  # slice -> a lazy view sequence

print("src[5]    ->", type(frame).__name__, np.asarray(frame.raw).shape)
print("src[5:15] ->", type(view).__name__, "with", len(view), "frames")

## Subsample, crop, pick a channel

All four axes at once, in one expression:

In [ ]:
print("every 2nd frame        ", len(src[::2]))
print("frames 3..22           ", len(src[3:23]))
print(
    "spatial crop           ",
    (src[:, 20:100, 30:130].size_h, src[:, 20:100, 30:130].size_w),
)
print("channel 0 only         ", src[..., 0].size_c)

composed = src[::2, 20:100, 30:130, 0]
print(
    "subsample + crop + chan",
    (composed.size_t, composed.size_h, composed.size_w, composed.size_c),
)

This is the single most useful habit in `acia`: **subsample before you compute.**
Segmentation and video rendering cost time proportional to frames and pixels, so
developing an analysis on `src[::20, 256:768, 256:768]` and only then turning the
knob back up turns a coffee break into a few seconds.

## Views are lazy and they compose

A view holds a reference to its parent and computes frames on demand. Nothing is
copied until you ask for pixels, and views of views are fine.

In [ ]:
v1 = src[::2]  # every 2nd frame
v2 = v1[1:]  # ... then drop the first of those
v3 = v2[:, :64, :64]  # ... then crop

for name, s in [("src", src), ("src[::2]", v1), ("...[1:]", v2), ("...crop", v3)]:
    print(f"{name:10} {type(s).__name__:24} T={len(s):3}  H={s.size_h}  W={s.size_w}")

## Channels

`to_channel(c)` is the readable form of `src[..., c]`, and works on every source
implementation.

In [ ]:
# a two-channel sequence: the disc, plus an inverted copy
two_channel = THWCSequenceSource(np.concatenate([stack, 255 - stack], axis=-1))

print("channels        ", two_channel.num_channels)
print("to_channel(1)   ", two_channel.to_channel(1).num_channels)
print("equivalent to   ", two_channel[..., 1].num_channels)

## When you do want it all in memory

`materialize()` walks the sequence once and returns a
{class}`~acia.segm.local.THWCSequenceSource` backed by a real `(T, H, W, C)`
array. Use it deliberately — after cropping and subsampling, not before.

In [ ]:
small = src[::4, 40:104, 40:104]
eager = small.materialize()

print(type(eager).__name__, "->", eager.image_stack.shape, eager.image_stack.dtype)

## What you learned

| You want | You write |
| --- | --- |
| one frame | `src[5]` |
| every 2nd frame | `src[::2]` |
| a frame range | `src[3:23]` |
| a spatial crop | `src[:, 100:200, 50:150]` |
| one channel | `src[..., 0]` or `src.to_channel(0)` |
| all of the above | `src[::2, 100:200, 50:150, 0]` |
| the whole thing as an array | `src.materialize()` |

Views are lazy, compose freely, and never copy pixels until asked.

Next: [3. Look at your data](03_look_at_your_data.ipynb) — turning a source into
something you can actually see.